# 현장건의 청크 임베딩: KURE-v1 (Colab GPU)
1. 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU** 선택
2. 아래 셀을 위에서부터 차례로 실행
3. 업로드할 파일: `exmnt_colab_input.json` (로컬 `data/processed/`에 있음, 약 13.6MB)
4. 마지막 셀이 `exmnt_colab_output.npz`를 내려받게 한다 → 로컬 `data/processed/`에 두고 `python pipeline/x4_exmnt_colab_import.py` 실행

입력 파일에는 현장건의 원문(개인정보는 가린 상태)이 들어 있다. 작업이 끝나면 런타임을 종료해 Colab에 남기지 않는다.

In [ ]:
!pip -q install sentence-transformers

In [ ]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "GPU 없음 - 런타임 유형을 확인하세요")

In [ ]:
from google.colab import files
up = files.upload()   # exmnt_colab_input.json 선택
assert "exmnt_colab_input.json" in up, "파일 이름을 확인하세요"

In [ ]:
import json, time, hashlib, numpy as np
from sentence_transformers import SentenceTransformer
src = json.load(open("exmnt_colab_input.json", encoding="utf-8"))
texts = src["chunks"]
sha = hashlib.sha256("\n\u241e\n".join(texts).encode("utf-8")).hexdigest()
assert len(texts) == src["n"] and sha == src["sha256"], "입력 파일이 손상됐습니다"
print(src["model"], len(texts), "청크, 최대 길이", src["max_len"])

BATCH = 32
t0 = time.time()
m = SentenceTransformer(src["model"], device="cuda")
m.max_seq_length = src["max_len"]   # 로컬 질문 임베딩과 같은 최대 길이
m.half()                             # fp16: 49번에서 fp32와 코사인 최소 0.9999 확인
order = np.argsort([len(t) for t in texts])   # 길이순으로 묶어 패딩을 줄인다
vecs = m.encode([texts[i] for i in order], batch_size=BATCH, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True)
out = np.empty_like(vecs)
out[order] = vecs
out = out.astype(np.float32)
norms = np.linalg.norm(out, axis=1)
print(out.shape, f"{time.time() - t0:.0f}초", "NaN:", bool(np.isnan(out).any()), "노름:", round(float(norms.min()), 4), "~", round(float(norms.max()), 4))
np.savez("exmnt_colab_output.npz", chunks=out, sha256=np.array(sha), model=np.array(src["model"]))

In [ ]:
files.download("exmnt_colab_output.npz")   # 약 51MB (12,445 x 1024 x 4바이트)